In [0]:
from pyspark.sql.functions import current_timestamp, col, count, when, input_file_name

In [0]:
import pandas as pd

url = "https://data.smartdublin.ie/dataset/33ec9fe2-4957-4e9a-ab55-c5e917c7a9ab/resource/03dd67ef-33b0-4102-8329-42c67fdbf53e/download/dublin-bikes_station_status_052026.csv"

pdf = pd.read_csv(url)

df = spark.createDataFrame(pdf)

In [0]:
df.write.mode("overwrite").option("header", "true").csv(
    "abfss://landing@dubmobdev001.dfs.core.windows.net/dublinbikes/year=2026/month=05/"
)

In [0]:
df = spark.read.csv(
    "abfss://landing@dubmobdev001.dfs.core.windows.net/dublinbikes/year=2026/month=05/",
    header=True,
    inferSchema=True
)

display(df)

In [0]:
total_rows = df.count()

for c in df.columns:
    null_count = df.filter(col(c).isNull()).count()
    print(f"{c}: {(null_count/total_rows)*100:.2f}%")

In [0]:
df = df.drop("short_name", "region_id")

In [0]:
df = df.withColumn(
    "processed_timestamp",
    current_timestamp()
)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(
    "urban_mobility.bronze.stations_raw"
)
